# Data Preparation Workflow

In [ ]:
# !pip install pandas scikit-learn

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

- charger les données en DataFrame

In [ ]:
path = input("Entrer le chemin du dataset CSV: ")
    

# vérifier l'existence du dossier ../data et valider le chemin du fichier CSV
default_dir = "../data"
default_path = os.path.join(default_dir, "movies.csv")

os.makedirs(default_dir, exist_ok=True)

# nettoyer la saisie utilisateur
path = (path or "").strip()

# si vide ou extension invalide -> utiliser le fichier par défaut
if not path or not path.lower().endswith(".csv"):
    path = default_path

# si le chemin choisi n'existe pas, tenter le fallback sur le fichier par défaut
if not os.path.exists(path):
    if path != default_path and os.path.exists(default_path):
        print(f"Chemin introuvable. Utilisation du fichier par défaut : {default_path}")
        path = default_path
    else:
        raise FileNotFoundError(f"Le fichier {os.path.basename(default_path)} n'existe pas dans le dossier {default_dir}")

In [ ]:
dataframe = pd.read_csv(path)

In [ ]:
dataframe.info()

In [ ]:
dataframe.head(3)

- traiter les données : fixer type des données (entiers, flottants, chaînes de caractères)

In [ ]:
# variables explicatives : rating, votes, popularity
dataframe["rating"] = pd.to_numeric(dataframe["rating"], errors='coerce').astype(float)

dataframe["votes"] = pd.to_numeric(dataframe["votes"], errors='coerce').astype(float)

- traiter les données : valeurs manquantes

In [ ]:
# supprimer les lignes avec des valeurs manquantes
print("Nombre de lignes avant suppression des valeurs manquantes :", len(dataframe))
print("Nombre de valeurs manquantes par colonne :")
print(dataframe.isnull().sum())
dataframe.dropna(inplace=True)
print("Nombre de lignes après suppression des valeurs manquantes :", len(dataframe))

- traiter les données : uniformisation des genres

In [ ]:
# uniformiser les genres : garder uniquement le genre principal (le premier genre listé)
print("Genres avant uniformisation :", dataframe['genre'].unique())
dataframe['genre'] = dataframe['genre'].apply(lambda x: x.split(',')[0].strip())
print("Genres après uniformisation :", dataframe['genre'].unique())
dataframe.head(3)

- convertir les genres (**valeur prédite**) en format numérique

In [ ]:
label_encoder = LabelEncoder()
dataframe['genre_encoded'] = label_encoder.fit_transform(dataframe['genre'])
print("Genres :", dataframe['genre'].unique())
print("Genres encodés :", dataframe['genre_encoded'].unique())

In [ ]:
dataframe.head(3)

In [ ]:
print(dataframe[["rating", "votes"]].dtypes)
print(dataframe[["rating", "votes"]].head())

- créer la feature **popularité** selon les critères suivants :
    + technique `somme pondérée` : *votes* x *rating*


In [ ]:
dataframe["popularity"] = dataframe["rating"] * dataframe["votes"]
dataframe.info()

In [ ]:
dataframe.head(3)

In [ ]:
# nombre de lignes par genre encodé
print("Nombre de lignes par genre encodé :")
print(dataframe['genre_encoded'].value_counts())

- traiter les données : traiter les corrélations entre les features

In [ ]:
correlation_data = dataframe[["rating", "votes", "popularity"]].corr()
correlation_data

On constate que 'popularity est très en corrélation avec 'votes' : 0.99 **> 0.8**.

Donc il **faut tester avec l'une ou l'autre de ces features** afin de constater l'impact de chacune sur les performances du modèle classifieur à entrainer.

In [ ]:
dataframe1 = dataframe[["rating", "votes", "genre_encoded"]]
dataframe2 = dataframe[["rating", "popularity", "genre_encoded"]]

### Exporter le DataFrame en CSV

In [ ]:
dataframe.to_csv('../data/movies_cleaned.csv', index=False)

dataframe1.to_csv('../data/movies_cleaned1.csv', index=False)
dataframe2.to_csv('../data/movies_cleaned2.csv', index=False)